In [ ]:
# python fake_tse_analysis.py --output-dir fake_tse_analysis_output

In [22]:
from pathlib import Path
import warnings

import nibabel as nib
import numpy as np
import pandas as pd

In [1]:
### COnfigs

# Tuple: (stim_session, sham_session)
SUBJECT_TABLE = {
    "sub-001": ("ses-04", "ses-06"),
    "sub-002": ("ses-04", "ses-06"),
    "sub-003": ("ses-06", "ses-04"),
    "sub-004": ("ses-06", "ses-04"),
   # "sub-005": ("ses-04", "ses-06"),
    "sub-010": ("ses-06", "ses-04"),
    "sub-014": ("ses-06", "ses-04"),
    "sub-015": ("ses-06", "ses-04"),
    "sub-016": ("ses-04", "ses-06"),
    "sub-017": ("ses-04", "ses-06"),
    "sub-018": ("ses-04", "ses-06"),
    #"sub-019": ("ses-06", "ses-04"),
    "sub-020": ("ses-04", "ses-06"),
    "sub-021": ("ses-06", "ses-04"),
    "sub-023": ("ses-04", "ses-06"),
    "sub-024": ("ses-06", "ses-04"),
    "sub-025": ("ses-04", "ses-06"),
    "sub-026": ("ses-04", "ses-06"),
    "sub-028": ("ses-06", "ses-04"),
    "sub-029": ("ses-04", "ses-06"),
    "sub-030": ("ses-06", "ses-04"),
    "sub-031": ("ses-06", "ses-04"),
    "sub-032": ("ses-04", "ses-06"),
    "sub-033": ("ses-06", "ses-04"),
    "sub-034": ("ses-04", "ses-06"),
    "sub-035": ("ses-06", "ses-04"),
    "sub-038": ("ses-06", "ses-04"),
    "sub-039": ("ses-06", "ses-04"),
    "sub-040": ("ses-04", "ses-06"),
    "sub-041": ("ses-04", "ses-06"),
    "sub-042": ("ses-06", "ses-04"),
    "sub-044": ("ses-04", "ses-06"),
    "sub-045": ("ses-04", "ses-06"),  # special case
    "sub-046": ("ses-06", "ses-04"),
    "sub-047": ("ses-06", "ses-04"),
    #"sub-048": ("ses-04", "ses-06"),
    "sub-052": ("ses-04", "ses-06"),
    "sub-053": ("ses-06", "ses-04"),
    "sub-054": ("ses-04", "ses-06"),
    "sub-055": ("ses-06", "ses-04"),
    "sub-056": ("ses-06", "ses-04"),
    "sub-057": ("ses-06", "ses-04"),
    "sub-058": ("ses-06", "ses-04"),
    "sub-059": ("ses-04", "ses-06"),
    "sub-060": ("ses-04", "ses-06"),
    "sub-061": ("ses-06", "ses-04"),
 #   "sub-063": ("ses-06", "ses-04"),
    "sub-064": ("ses-04", "ses-06"),
    "sub-065": ("ses-06", "ses-04"),
    "sub-066": ("ses-06", "ses-04"),
    "sub-067": ("ses-06", "ses-04"),
    "sub-068": ("ses-04", "ses-06"),
    "sub-069": ("ses-06", "ses-04"),
    "sub-070": ("ses-04", "ses-06"),
    "sub-071": ("ses-06", "ses-04"),
    "sub-072": ("ses-04", "ses-06"),
    "sub-073": ("ses-04", "ses-06"),
    "sub-075": ("ses-04", "ses-06"),
    "sub-076": ("ses-04", "ses-06"),
    "sub-077": ("ses-04", "ses-06"),
    "sub-078": ("ses-04", "ses-06"),
    "sub-080": ("ses-06", "ses-04"),
    "sub-081": ("ses-04", "ses-06"),
    "sub-082": ("ses-04", "ses-06"),
    "sub-084": ("ses-06", "ses-04"),
}

BASELINE_SESSIONS = ["ses01", "ses02"]
EXPERIMENTAL_SESSIONS = ["ses03", "ses04", "ses05", "ses06"]

In [ ]:
"""
The dataframe should have:
sub00 | ses00 | z_axis | lc_cont_mean_sides | left_lc_cont | right_lc_cont

1. Then from that dataframe I get per sub/per session:

z_range_axis
lc_cont_mean_across_z_axis

min_lc_cont_mean_sides
max_lc_cont_mean_sides
sd_lc_cont_mean_sides

min_left_lc_cont
max_left_lc_cont
sd_left_lc_cont

min_right_lc_cont
max_right_lc_cont
sd_right_lc_cont

peak_z_axis

2. Plots:
a. 3 Violin plots with the t-test (pairs and baseline) *use the table to check if pair01 or pair02 is stims or sham for each subs
b. Compute the chage in stims and sham for each subs *use the table to check if pair01 or pair02 is stims or sham for each subs
"""

In [23]:
# we will load or LC_mask or lc_ft_cluster or any other mask
def _load_mask(mask_path: Path) -> tuple[nib.Nifti1Image, np.ndarray]:
    img = nib.load(str(mask_path))
    mask = np.asarray(img.dataobj) > 0
    return img, mask

# from the whole lc mask create two masks, each one for one lc hemesphere
def _create_hemisphere_masks(
    shape: tuple[int, int, int],
    affine: np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:

    voxel_indices = np.indices(shape)
    flat_indices = voxel_indices.reshape(3, -1).T

    world_coordinates = nib.affines.apply_affine(
        affine,
        flat_indices,
    )

    x_coordinates = world_coordinates[:, 0].reshape(shape)

    left_mask = x_coordinates < 0
    right_mask = x_coordinates > 0

    return left_mask, right_mask


# compute the lc_contrast -- TODO: add the other types of computation
# per hemispehre
def _compute_lc_contrast(
    tse_slice: np.ndarray,
    lc_slice_mask: np.ndarray,
    dpt_slice_mask: np.ndarray,
) -> float:

    if not np.any(lc_slice_mask):
        return np.nan

    if not np.any(dpt_slice_mask):
        return np.nan

    lc_values = tse_slice[lc_slice_mask]
    dpt_values = tse_slice[dpt_slice_mask]

    lc_values = lc_values[np.isfinite(lc_values)]
    dpt_values = dpt_values[np.isfinite(dpt_values)]

    if lc_values.size == 0 or dpt_values.size == 0:
        return np.nan

    lc_mean = float(np.max(lc_values))
    dpt_mean = float(np.max(dpt_values))

    if not np.isfinite(dpt_mean) or np.isclose(dpt_mean, 0):
        return np.nan

    return (lc_mean - dpt_mean) / dpt_mean



In [24]:
# create the dataframe with the values
"""
The dataframe should have:
sub00 | ses00 | z_axis | lc_cont_mean_sides | left_lc_cont | right_lc_cont
"""

def build_dataframe(
    root_dir,
    session: str = "ses001",
    output_csv=None,
    include_empty_slices: bool = False,
    strict: bool = False,
) -> pd.DataFrame:
    
    root_dir = Path(root_dir)

    tse_root = root_dir / "step6_tse_mni"
    masks_root = root_dir / "step7_masks_grid"
    cluster_root = root_dir / "step8_contrast"

    if not tse_root.exists():
        raise FileNotFoundError(
            f"TSE root directory does not exist: {tse_root}"
        )

    subject_dirs = sorted(
        path
        for path in tse_root.glob("sub*")
        if path.is_dir()
    )

    if not subject_dirs:
        raise FileNotFoundError(
            f"No subject directories found in: {tse_root}"
        )

    rows = []
    processed_subjects = []
    skipped_subjects = []

    for subject_dir in subject_dirs:
        subject = subject_dir.name

        tse_path = (
            tse_root
            / subject
            / session
            / "tse_in_MNI_brainstem_0p5mm.nii.gz"
        )

        original_lc_path = (
            masks_root
            / subject
            / session
            / "LC_mask_grid.nii.gz"
        )

        dpt_path = (
            masks_root
            / subject
            / session
            / "DPT_mask_grid.nii.gz"
        )

        cluster_lc_path = (
            cluster_root
            / subject
            / session
            / "lc_ft_cluster_mask.nii.gz"
        )

        required_files = {
            "TSE": tse_path,
            "original LC mask": original_lc_path,
            "DPT mask": dpt_path,
            "cluster LC mask": cluster_lc_path,
        }

        missing_files = {
            name: path
            for name, path in required_files.items()
            if not path.exists()
        }

        if missing_files:
            message = (
                f"{subject}/{session} has missing files:\n"
                + "\n".join(
                    f"  {name}: {path}"
                    for name, path in missing_files.items()
                )
            )

            if strict:
                raise FileNotFoundError(message)

            warnings.warn(message)
            skipped_subjects.append(subject)
            continue

        try:

            tse_img = nib.load(str(tse_path))
            tse_data = np.asarray(
                tse_img.dataobj,
                dtype=np.float32,
            )

            if tse_data.ndim != 3:
                raise ValueError(
                    f"Expected a 3D TSE image, got shape {tse_data.shape}"
                )

            # Load the three masks.
            original_lc_img, original_lc_mask = _load_mask(
                original_lc_path
            )

            cluster_lc_img, cluster_lc_mask = _load_mask(
                cluster_lc_path
            )

            dpt_img, dpt_mask = _load_mask(dpt_path)

            # get left and right hemispehere
            left_space, right_space = _create_hemisphere_masks(
                shape=tse_data.shape,
                affine=tse_img.affine,
            )

            if include_empty_slices:
                z_indices = range(tse_data.shape[2])

            else:
                # Include the union of slices represented by either LC mask.
                either_lc_mask = original_lc_mask | cluster_lc_mask

                z_indices = np.flatnonzero(
                    np.any(either_lc_mask, axis=(0, 1))
                )

            for z_axis in z_indices:
                z_axis = int(z_axis)

                tse_slice = tse_data[:, :, z_axis]

                left_space_slice = left_space[:, :, z_axis]
                right_space_slice = right_space[:, :, z_axis]

                dpt_slice = dpt_mask[:, :, z_axis]

                left_dpt_slice = (
                    dpt_slice
                    & left_space_slice
                )

                right_dpt_slice = (
                    dpt_slice
                    & right_space_slice
                )

                original_lc_slice = original_lc_mask[:, :, z_axis]

                original_left_lc_slice = (
                    original_lc_slice
                    & left_space_slice
                )

                original_right_lc_slice = (
                    original_lc_slice
                    & right_space_slice
                )

                original_left_lc_cont = _compute_lc_contrast(
                    tse_slice=tse_slice,
                    lc_slice_mask=original_left_lc_slice,
                    dpt_slice_mask=left_dpt_slice,
                )

                original_right_lc_cont = _compute_lc_contrast(
                    tse_slice=tse_slice,
                    lc_slice_mask=original_right_lc_slice,
                    dpt_slice_mask=right_dpt_slice,
                )

                #original_lc_cont_mean_sides = _mean_valid_sides(
                #    original_left_lc_cont,
                #    original_right_lc_cont,
                #)

                cluster_lc_slice = cluster_lc_mask[:, :, z_axis]

                cluster_left_lc_slice = (
                    cluster_lc_slice
                    & left_space_slice
                )

                cluster_right_lc_slice = (
                    cluster_lc_slice
                    & right_space_slice
                )

                cluster_left_lc_cont = _compute_lc_contrast(
                    tse_slice=tse_slice,
                    lc_slice_mask=cluster_left_lc_slice,
                    dpt_slice_mask=left_dpt_slice,
                )

                cluster_right_lc_cont = _compute_lc_contrast(
                    tse_slice=tse_slice,
                    lc_slice_mask=cluster_right_lc_slice,
                    dpt_slice_mask=right_dpt_slice,
                )

                #cluster_lc_cont_mean_sides = _mean_valid_sides(
                #    cluster_left_lc_cont,
                #    cluster_right_lc_cont,
                #)

                rows.append(
                    {
                        "subject": subject,
                        "session": session,
                        "z_axis": z_axis,

                        #"original_lc_cont_mean_sides":
                        #    original_lc_cont_mean_sides,
                        "original_left_lc_cont":
                            original_left_lc_cont,
                        "original_right_lc_cont":
                            original_right_lc_cont,

                        #"cluster_lc_cont_mean_sides":
                        #    cluster_lc_cont_mean_sides,
                        "cluster_left_lc_cont":
                            cluster_left_lc_cont,
                        "cluster_right_lc_cont":
                            cluster_right_lc_cont,
                    }
                )

            processed_subjects.append(subject)

        except Exception as error:
            message = (
                f"Could not process {subject}/{session}: {error}"
            )

            if strict:
                raise RuntimeError(message) from error

            warnings.warn(message)
            skipped_subjects.append(subject)

    column_order = [
        "subject",
        "session",
        "z_axis",
        #"original_lc_cont_mean_sides",
        "original_left_lc_cont",
        "original_right_lc_cont",
        #"cluster_lc_cont_mean_sides",
        "cluster_left_lc_cont",
        "cluster_right_lc_cont",
    ]

    result_df = pd.DataFrame(
        rows,
        columns=column_order,
    )

    if not result_df.empty:
        result_df = (
            result_df
            .sort_values(
                ["subject", "session", "z_axis"]
            )
            .reset_index(drop=True)
        )

    if output_csv is not None:
        output_csv = Path(output_csv)
        output_csv.parent.mkdir(
            parents=True,
            exist_ok=True,
        )

        result_df.to_csv(
            output_csv,
            index=False,
        )

        print(f"Saved CSV: {output_csv}")

    print(f"Processed subjects: {len(processed_subjects)}")
    print(f"Skipped subjects: {len(set(skipped_subjects))}")
    print(f"Dataframe rows: {len(result_df)}")

    return result_df

In [29]:
subj002 = build_dataframe(
    root_dir="/home/maria/Documents/data/hiwi_sample/tmp-maria-exp11_step08_changed_subj001",
    session="ses001",
    output_csv="/home/maria/Documents/projects/mri_hiwi/results/out_subs002_ses001.csv",
    include_empty_slices=False,
    strict=False,
)

Saved CSV: /home/maria/Documents/projects/mri_hiwi/results/out_subs002_ses001.csv
Processed subjects: 1
Skipped subjects: 0
Dataframe rows: 28


In [30]:
subj017 = build_dataframe(
    root_dir="/home/maria/Documents/data/hiwi_sample/tmp-maria-exp11_step08_changed_subj017",
    session="ses001",
    output_csv="/home/maria/Documents/projects/mri_hiwi/results/out_subs017_ses001.csv",
    include_empty_slices=False,
    strict=False,
)

Saved CSV: /home/maria/Documents/projects/mri_hiwi/results/out_subs017_ses001.csv
Processed subjects: 1
Skipped subjects: 0
Dataframe rows: 28


In [38]:
subj002["original_mean_lc_cont"] = subj002[
    ["original_left_lc_cont", "original_right_lc_cont"]
].mean(axis=1, skipna=False)

subj002["cluster_mean_lc_cont"] = subj002[
    ["cluster_left_lc_cont", "cluster_right_lc_cont"]
].mean(axis=1, skipna=False)

subj002

,subject,session,z_axis,original_left_lc_cont,original_right_lc_cont,cluster_left_lc_cont,cluster_right_lc_cont,original_mean_lc_cont,cluster_mean_lc_cont
0,sub002,ses001,84,-0.139070,NaN,0.046618,NaN,NaN,NaN
1,sub002,ses001,85,-0.091093,NaN,0.082923,NaN,NaN,NaN
2,sub002,ses001,86,-0.058987,NaN,0.072525,NaN,NaN,NaN
3,sub002,ses001,87,0.027935,NaN,0.038712,NaN,NaN,NaN
4,sub002,ses001,88,-0.022524,-0.031653,0.058028,0.057725,-0.027088,0.057876
5,sub002,ses001,89,0.021337,0.055825,0.092014,0.097140,0.038581,0.094577
6,sub002,ses001,90,0.057968,0.075624,0.057968,0.227761,0.066796,0.142864
7,sub002,ses001,91,0.055670,0.202362,0.122845,0.202362,0.129016,0.162603
8,sub002,ses001,92,0.235475,0.192545,0.259015,0.192545,0.214010,0.225780
9,sub002,ses001,93,0.214788,0.307254,0.295706,0.307254,0.261021,0.301480


In [39]:
subj017["original_mean_lc_cont"] = subj017[
    ["original_left_lc_cont", "original_right_lc_cont"]
].mean(axis=1, skipna=False)

subj017["cluster_mean_lc_cont"] = subj017[
    ["cluster_left_lc_cont", "cluster_right_lc_cont"]
].mean(axis=1, skipna=False)

subj017

,subject,session,z_axis,original_left_lc_cont,original_right_lc_cont,cluster_left_lc_cont,cluster_right_lc_cont,original_mean_lc_cont,cluster_mean_lc_cont
0,sub017,ses001,84,-0.066697,NaN,0.063046,NaN,NaN,NaN
1,sub017,ses001,85,-0.076869,NaN,-0.043220,NaN,NaN,NaN
2,sub017,ses001,86,-0.088184,NaN,0.059129,NaN,NaN,NaN
3,sub017,ses001,87,-0.099594,NaN,-0.049646,NaN,NaN,NaN
4,sub017,ses001,88,0.043325,0.008230,0.067799,0.076159,0.025778,0.071979
5,sub017,ses001,89,0.073644,0.072746,0.110541,0.098593,0.073195,0.104567
6,sub017,ses001,90,0.138860,0.083182,0.138860,0.083182,0.111021,0.111021
7,sub017,ses001,91,0.131148,0.068652,0.131148,0.068652,0.099900,0.099900
8,sub017,ses001,92,0.137472,0.088626,0.137472,0.088626,0.113049,0.113049
9,sub017,ses001,93,0.173688,0.077626,0.173688,0.077626,0.125657,0.125657


In [15]:
from pathlib import Path


root_dir = Path(
    "/media/maria/A91A-44D3/mri_hiwi/"
    "exp_all_subs_ses001"
)

all_subjects_df = build_all_subjects_lc_contrast_dataframe(
    root_dir=root_dir,
    session="ses001",
    output_csv=(
        root_dir
        / "all_subjects_original_and_cluster_lc_contrast_per_z.csv"
    ),
    include_empty_slices=False,
    strict=False,
)

print(all_subjects_df.head(20))

/tmp/ipykernel_74030/1149530013.py:264: UserWarning: sub005/ses001 has missing files:
  cluster LC mask: /media/maria/A91A-44D3/mri_hiwi/exp_all_subs_ses001/step8_contrast/sub005/ses001/lc_ft_cluster_mask.nii.gz
  warnings.warn(message)
/tmp/ipykernel_74030/1149530013.py:264: UserWarning: sub046/ses001 has missing files:
  cluster LC mask: /media/maria/A91A-44D3/mri_hiwi/exp_all_subs_ses001/step8_contrast/sub046/ses001/lc_ft_cluster_mask.nii.gz
  warnings.warn(message)
/tmp/ipykernel_74030/1149530013.py:264: UserWarning: sub054/ses001 has missing files:
  cluster LC mask: /media/maria/A91A-44D3/mri_hiwi/exp_all_subs_ses001/step8_contrast/sub054/ses001/lc_ft_cluster_mask.nii.gz
  warnings.warn(message)
/tmp/ipykernel_74030/1149530013.py:264: UserWarning: sub065/ses001 has missing files:
  cluster LC mask: /media/maria/A91A-44D3/mri_hiwi/exp_all_subs_ses001/step8_contrast/sub065/ses001/lc_ft_cluster_mask.nii.gz
  warnings.warn(message)
/tmp/ipykernel_74030/1149530013.py:264: UserWarning:

Saved CSV: /media/maria/A91A-44D3/mri_hiwi/exp_all_subs_ses001/all_subjects_original_and_cluster_lc_contrast_per_z.csv
Processed subjects: 58
Skipped subjects: 6
Dataframe rows: 1624
   subject session  z_axis  original_lc_cont_mean_sides  \
0   sub001  ses001      84                     2.442938   
1   sub001  ses001      85                     2.099012   
2   sub001  ses001      86                     1.563324   
3   sub001  ses001      87                     1.079692   
4   sub001  ses001      88                     0.930713   
5   sub001  ses001      89                     0.678171   
6   sub001  ses001      90                     0.523378   
7   sub001  ses001      91                     0.451910   
8   sub001  ses001      92                     0.406724   
9   sub001  ses001      93                     0.411425   
10  sub001  ses001      94                     0.438109   
11  sub001  ses001      95                     0.474470   
12  sub001  ses001      96                     0.5

/tmp/ipykernel_74030/1149530013.py:264: UserWarning: sub084/ses001 has missing files:
  cluster LC mask: /media/maria/A91A-44D3/mri_hiwi/exp_all_subs_ses001/step8_contrast/sub084/ses001/lc_ft_cluster_mask.nii.gz
  warnings.warn(message)


In [21]:
all_subjects_df[all_subjects_df["subject"]=="sub001"]

,subject,session,z_axis,original_lc_cont_mean_sides,original_left_lc_cont,original_right_lc_cont,cluster_lc_cont_mean_sides,cluster_left_lc_cont,cluster_right_lc_cont
0,sub001,ses001,84,2.442938,2.442938,NaN,2.752209,2.752209,NaN
1,sub001,ses001,85,2.099012,2.099012,NaN,2.465776,2.465776,NaN
2,sub001,ses001,86,1.563324,1.563324,NaN,1.680382,1.680382,NaN
3,sub001,ses001,87,1.079692,1.079692,NaN,1.248610,1.248610,NaN
4,sub001,ses001,88,0.930713,0.733995,1.127431,1.227080,0.908117,1.546043
5,sub001,ses001,89,0.678171,0.531590,0.824752,0.958064,0.697721,1.218408
6,sub001,ses001,90,0.523378,0.411987,0.634770,0.794413,0.581306,1.007519
7,sub001,ses001,91,0.451910,0.293455,0.610365,0.755232,0.472098,1.038366
8,sub001,ses001,92,0.406724,0.252051,0.561397,0.705691,0.444207,0.967175
9,sub001,ses001,93,0.411425,0.284748,0.538101,0.693875,0.475512,0.912237


In [ ]:
"/home/maria/Documents/data/hiwi_sample/tmp-maria-exp11_step08_changed_subj001"

In [8]:
### We dont have the data yet, so lets create a fake dataframe
from pathlib import Path
import argparse

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import ttest_rel

def normalize_session(session: str) -> str:
    """Convert 'ses-04' or 'ses04' to 'ses04'."""
    digits = "".join(character for character in str(session) if character.isdigit())
    if not digits:
        raise ValueError(f"Could not parse session: {session!r}")
    return f"ses{int(digits):02d}"

def create_fake_tse_data(
    subject_table: dict[str, tuple[str, str]],
    seed: int = 20260724,
    baseline_mean: float = 500.0,
    between_subject_sd: float = 45.0,
    evening_to_evening_sd: float = 5.0,
    overnight_noise_sd: float = 5.0,
    stimulation_effect_mean: float = 8.0,
    stimulation_effect_sd: float = 5.0,
    missing_experimental_fraction: float = 0.08,
    missing_adaptation_fraction: float = 0.08,
) -> pd.DataFrame:

    rng = np.random.default_rng(seed)
    rows: list[dict[str, float | str]] = []

    for subject, (stim_end_raw, _) in subject_table.items():
        stim_end = normalize_session(stim_end_raw)

        subject_baseline = rng.normal(baseline_mean, between_subject_sd)

        # Adaptation night.
        ses01 = subject_baseline + rng.normal(0.0, evening_to_evening_sd)

        ses02 = (
            ses01
            + rng.normal(0.0, overnight_noise_sd)
        )

        # Two experimental evenings should be close to one another.
        common_evening = subject_baseline + rng.normal(0.0, evening_to_evening_sd)
        ses03 = common_evening + rng.normal(0.0, evening_to_evening_sd / 2)
        ses05 = common_evening + rng.normal(0.0, evening_to_evening_sd / 2)

        # Stimulation-specific extra effect.
        stimulation_effect = rng.normal(
            stimulation_effect_mean,
            stimulation_effect_sd,
        )

        if stim_end == "ses04":
            # ses03 -> ses04 is stimulation.
            ses04 = (
                ses03
                + stimulation_effect
                + rng.normal(0.0, overnight_noise_sd)
            )

            # ses05 -> ses06 is sham.
            ses06 = (
                ses05
                + rng.normal(0.0, overnight_noise_sd)
            )

        elif stim_end == "ses06":
            # ses03 -> ses04 is sham.
            ses04 = (
                ses03
                + rng.normal(0.0, overnight_noise_sd)
            )

            # ses05 -> ses06 is stimulation.
            ses06 = (
                ses05
                + stimulation_effect
                + rng.normal(0.0, overnight_noise_sd)
            )

        else:
            raise ValueError(
                f"Unexpected stimulation endpoint for {subject}: {stim_end}"
            )

        rows.append(
            {
                "subject": subject,
                "ses01": ses01,
                "ses02": ses02,
                "ses03": ses03,
                "ses04": ses04,
                "ses05": ses05,
                "ses06": ses06,
            }
        )

    data = (
        pd.DataFrame(rows)
        .sort_values("subject")
        .reset_index(drop=True)
    )

    n_experimental_missing = max(
        1,
        round(len(data) * missing_experimental_fraction),
    )
    experimental_indices = rng.choice(
        data.index,
        size=n_experimental_missing,
        replace=False,
    )

    for row_index in experimental_indices:
        missing_session = rng.choice(EXPERIMENTAL_SESSIONS)
        data.loc[row_index, missing_session] = np.nan

    remaining_indices = data.index.difference(experimental_indices)
    n_adaptation_missing = max(
        1,
        round(len(data) * missing_adaptation_fraction),
    )
    adaptation_indices = rng.choice(
        remaining_indices,
        size=n_adaptation_missing,
        replace=False,
    )

    for row_index in adaptation_indices:
        missing_session = rng.choice(["ses01", "ses02"])
        data.loc[row_index, missing_session] = np.nan


In [9]:
create_fake_tse_data(subject_table=SUBJECT_TABLE)

In [10]:

#!/usr/bin/env python3
"""
Create realistic fake TSE intensity data and run the requested analysis.

Model used for the fake data
----------------------------
1. Evening-to-evening values are similar:
       ses03 and ses05 differ only slightly.

2. There is a small common overnight/circadian change:
       morning = evening + circadian_change

3. Stimulation adds an extra overnight effect:
       stimulation morning = stimulation evening
                             + circadian_change
                             + stimulation_effect

4. Sham contains only the circadian component plus noise.

5. Subjects are filtered using ses03, ses04, ses05, ses06 only.
   Missing ses01/ses02 does not exclude a subject.

Outputs
-------
- fake_tse_values.csv
- included_complete_ses03_ses06.csv
- excluded_subjects.csv
- condition_long.csv
- change_scores.csv
- statistics.txt
- adaptation_violin.png
- stimulation_violin.png
- sham_violin.png
- stimulation_change_distribution.png
- sham_change_distribution.png
- paired_change_plot.png
"""

from __future__ import annotations

from pathlib import Path
import argparse

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import ttest_rel


def normalize_session(session: str) -> str:
    """Convert 'ses-04' or 'ses04' to 'ses04'."""
    digits = "".join(character for character in str(session) if character.isdigit())
    if not digits:
        raise ValueError(f"Could not parse session: {session!r}")
    return f"ses{int(digits):02d}"


def create_fake_tse_data(
    subject_table: dict[str, tuple[str, str]],
    *,
    seed: int = 20260724,
    baseline_mean: float = 500.0,
    between_subject_sd: float = 45.0,
    evening_to_evening_sd: float = 5.0,
    circadian_mean: float = 6.0,
    circadian_between_subject_sd: float = 4.0,
    overnight_noise_sd: float = 5.0,
    stimulation_effect_mean: float = 8.0,
    stimulation_effect_sd: float = 5.0,
    missing_experimental_fraction: float = 0.08,
    missing_adaptation_fraction: float = 0.08,
) -> pd.DataFrame:
    """
    Generate fake data with small evening-to-evening differences and a
    modest overnight change.

    Parameters can be adjusted to make the fake effect weaker or stronger.
    """
    rng = np.random.default_rng(seed)
    rows: list[dict[str, float | str]] = []

    for subject, (stim_end_raw, _) in subject_table.items():
        stim_end = normalize_session(stim_end_raw)

        # Stable subject-specific intensity level.
        subject_baseline = rng.normal(baseline_mean, between_subject_sd)

        # Adaptation night.
        ses01 = subject_baseline + rng.normal(0.0, evening_to_evening_sd)
        adaptation_circadian = rng.normal(
            circadian_mean,
            circadian_between_subject_sd,
        )
        ses02 = (
            ses01
            + adaptation_circadian
            + rng.normal(0.0, overnight_noise_sd)
        )

        # Two experimental evenings should be close to one another.
        common_evening = subject_baseline + rng.normal(0.0, evening_to_evening_sd)
        ses03 = common_evening + rng.normal(0.0, evening_to_evening_sd / 2)
        ses05 = common_evening + rng.normal(0.0, evening_to_evening_sd / 2)

        # Subject-specific circadian/overnight component.
        circadian_change = rng.normal(
            circadian_mean,
            circadian_between_subject_sd,
        )

        # Stimulation-specific extra effect.
        stimulation_effect = rng.normal(
            stimulation_effect_mean,
            stimulation_effect_sd,
        )

        if stim_end == "ses04":
            # ses03 -> ses04 is stimulation.
            ses04 = (
                ses03
                + circadian_change
                + stimulation_effect
                + rng.normal(0.0, overnight_noise_sd)
            )

            # ses05 -> ses06 is sham.
            ses06 = (
                ses05
                + circadian_change
                + rng.normal(0.0, overnight_noise_sd)
            )

        elif stim_end == "ses06":
            # ses03 -> ses04 is sham.
            ses04 = (
                ses03
                + circadian_change
                + rng.normal(0.0, overnight_noise_sd)
            )

            # ses05 -> ses06 is stimulation.
            ses06 = (
                ses05
                + circadian_change
                + stimulation_effect
                + rng.normal(0.0, overnight_noise_sd)
            )

        else:
            raise ValueError(
                f"Unexpected stimulation endpoint for {subject}: {stim_end}"
            )

        rows.append(
            {
                "subject": subject,
                "ses01": ses01,
                "ses02": ses02,
                "ses03": ses03,
                "ses04": ses04,
                "ses05": ses05,
                "ses06": ses06,
            }
        )

    data = (
        pd.DataFrame(rows)
        .sort_values("subject")
        .reset_index(drop=True)
    )

    # Add missing experimental values. These subjects must be excluded.
    n_experimental_missing = max(
        1,
        round(len(data) * missing_experimental_fraction),
    )
    experimental_indices = rng.choice(
        data.index,
        size=n_experimental_missing,
        replace=False,
    )

    for row_index in experimental_indices:
        missing_session = rng.choice(EXPERIMENTAL_SESSIONS)
        data.loc[row_index, missing_session] = np.nan

    # Add adaptation-only missingness. These subjects remain eligible.
    remaining_indices = data.index.difference(experimental_indices)
    n_adaptation_missing = max(
        1,
        round(len(data) * missing_adaptation_fraction),
    )
    adaptation_indices = rng.choice(
        remaining_indices,
        size=n_adaptation_missing,
        replace=False,
    )

    for row_index in adaptation_indices:
        missing_session = rng.choice(["ses01", "ses02"])
        data.loc[row_index, missing_session] = np.nan

    return data


def filter_complete_experimental(
    raw_df: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Keep subjects with valid values in ses03-ses06.

    ses01 and ses02 do not determine inclusion.
    """
    data = raw_df.copy()

    for session in [f"ses0{i}" for i in range(1, 7)]:
        data[session] = pd.to_numeric(data[session], errors="coerce")

    data = data.replace([np.inf, -np.inf], np.nan)

    in_subject_table = data["subject"].isin(SUBJECT_TABLE)
    complete_experimental = (
        data[EXPERIMENTAL_SESSIONS]
        .notna()
        .all(axis=1)
    )

    keep = in_subject_table & complete_experimental

    included = data.loc[keep].copy().reset_index(drop=True)
    excluded = data.loc[~keep].copy()

    def get_exclusion_reason(row: pd.Series) -> str:
        reasons: list[str] = []

        if row["subject"] not in SUBJECT_TABLE:
            reasons.append("not_in_subject_table")

        missing_sessions = [
            session
            for session in EXPERIMENTAL_SESSIONS
            if pd.isna(row[session])
        ]

        if missing_sessions:
            reasons.append(
                "missing_" + "_".join(missing_sessions)
            )

        return ";".join(reasons) if reasons else "unknown"

    excluded["exclusion_reason"] = excluded.apply(
        get_exclusion_reason,
        axis=1,
    )

    return included, excluded.reset_index(drop=True)


def create_condition_long_dataframe(
    included_df: pd.DataFrame,
) -> pd.DataFrame:
    """
    Create one row per subject, condition, and time point.

    SUBJECT_TABLE is used to decide which experimental pair is stimulation
    and which is sham.
    """
    pair_from_end_session = {
        "ses04": ("ses03", "ses04"),
        "ses06": ("ses05", "ses06"),
    }

    rows: list[dict[str, object]] = []

    for _, row in included_df.iterrows():
        subject = row["subject"]

        stim_end_raw, sham_end_raw = SUBJECT_TABLE[subject]
        stim_end = normalize_session(stim_end_raw)
        sham_end = normalize_session(sham_end_raw)

        stim_evening, stim_morning = pair_from_end_session[stim_end]
        sham_evening, sham_morning = pair_from_end_session[sham_end]

        session_mapping = [
            ("adaptation", "evening", "ses01"),
            ("adaptation", "morning", "ses02"),
            ("stimulation", "evening", stim_evening),
            ("stimulation", "morning", stim_morning),
            ("sham", "evening", sham_evening),
            ("sham", "morning", sham_morning),
        ]

        for condition, time, original_session in session_mapping:
            rows.append(
                {
                    "subject": subject,
                    "condition": condition,
                    "time": time,
                    "original_session": original_session,
                    "tse_intensity": row[original_session],
                }
            )

    return pd.DataFrame(rows)


def compute_change_scores(
    included_df: pd.DataFrame,
) -> pd.DataFrame:
    """
    Compute morning minus evening for stimulation and sham.
    """
    rows: list[dict[str, object]] = []

    for _, row in included_df.iterrows():
        subject = row["subject"]

        stim_end_raw, sham_end_raw = SUBJECT_TABLE[subject]
        stim_end = normalize_session(stim_end_raw)
        sham_end = normalize_session(sham_end_raw)

        delta_pair_04 = row["ses04"] - row["ses03"]
        delta_pair_06 = row["ses06"] - row["ses05"]

        delta_from_end_session = {
            "ses04": delta_pair_04,
            "ses06": delta_pair_06,
        }

        delta_stimulation = delta_from_end_session[stim_end]
        delta_sham = delta_from_end_session[sham_end]

        rows.append(
            {
                "subject": subject,
                "stim_pair": (
                    "ses03->ses04"
                    if stim_end == "ses04"
                    else "ses05->ses06"
                ),
                "sham_pair": (
                    "ses03->ses04"
                    if sham_end == "ses04"
                    else "ses05->ses06"
                ),
                "delta_pair_04": delta_pair_04,
                "delta_pair_06": delta_pair_06,
                "delta_stimulation": delta_stimulation,
                "delta_sham": delta_sham,
                "stim_minus_sham": (
                    delta_stimulation - delta_sham
                ),
            }
        )

    return pd.DataFrame(rows)


def plot_condition_violin(
    long_df: pd.DataFrame,
    condition: str,
    output_path: Path,
) -> None:
    subset = long_df[long_df["condition"] == condition]

    evening = (
        subset.loc[
            subset["time"] == "evening",
            "tse_intensity",
        ]
        .dropna()
        .to_numpy(float)
    )

    morning = (
        subset.loc[
            subset["time"] == "morning",
            "tse_intensity",
        ]
        .dropna()
        .to_numpy(float)
    )

    fig, ax = plt.subplots(figsize=(6.5, 5.0))

    ax.violinplot(
        [evening, morning],
        positions=[1, 2],
        showmeans=True,
        showmedians=True,
    )

    rng = np.random.default_rng(123)

    for x_position, values in zip(
        [1, 2],
        [evening, morning],
    ):
        jitter = rng.normal(0.0, 0.035, size=len(values))

        ax.scatter(
            np.full(len(values), x_position) + jitter,
            values,
            alpha=0.55,
            s=18,
        )

    ax.set_xticks([1, 2], ["Evening", "Morning"])
    ax.set_ylabel("TSE intensity")
    ax.set_title(
        f"{condition.capitalize()}: evening versus morning"
    )
    ax.grid(axis="y", alpha=0.25)

    fig.tight_layout()
    fig.savefig(output_path, dpi=180)
    plt.close(fig)


def plot_change_distribution(
    values: pd.Series,
    title: str,
    output_path: Path,
) -> None:
    clean = values.dropna().to_numpy(float)

    fig, ax = plt.subplots(figsize=(6.5, 5.0))

    ax.hist(
        clean,
        bins="auto",
        density=False,
        alpha=0.65,
    )

    ax.axvline(
        0.0,
        linestyle="--",
        linewidth=1.2,
        label="No change",
    )

    ax.axvline(
        clean.mean(),
        linewidth=2,
        label=f"Mean = {clean.mean():.2f}",
    )

    ax.set_xlabel("Morning - evening TSE intensity")
    ax.set_ylabel("Number of subjects")
    ax.set_title(title)
    ax.grid(axis="y", alpha=0.25)
    ax.legend()

    fig.tight_layout()
    fig.savefig(output_path, dpi=180)
    plt.close(fig)


def plot_paired_change(
    changes_df: pd.DataFrame,
    output_path: Path,
) -> None:
    stimulation = changes_df["delta_stimulation"].to_numpy(float)
    sham = changes_df["delta_sham"].to_numpy(float)

    fig, ax = plt.subplots(figsize=(6.5, 5.2))

    for stimulation_value, sham_value in zip(
        stimulation,
        sham,
    ):
        ax.plot(
            [1, 2],
            [stimulation_value, sham_value],
            alpha=0.20,
            linewidth=0.9,
        )

    ax.scatter(
        np.ones(len(stimulation)),
        stimulation,
        alpha=0.65,
        s=22,
    )

    ax.scatter(
        np.full(len(sham), 2),
        sham,
        alpha=0.65,
        s=22,
    )

    ax.axhline(
        0.0,
        linestyle="--",
        linewidth=1.2,
    )

    ax.set_xticks([1, 2], ["Stimulation", "Sham"])
    ax.set_ylabel("Morning - evening TSE intensity")
    ax.set_title("Paired stimulation versus sham change")
    ax.grid(axis="y", alpha=0.25)

    fig.tight_layout()
    fig.savefig(output_path, dpi=180)
    plt.close(fig)


def run_analysis(
    output_dir: str | Path,
    *,
    seed: int = 20260724,
) -> None:
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    raw_df = create_fake_tse_data(
        SUBJECT_TABLE,
        seed=seed,
    )

    included_df, excluded_df = filter_complete_experimental(
        raw_df
    )

    long_df = create_condition_long_dataframe(
        included_df
    )

    changes_df = compute_change_scores(
        included_df
    )

    # Paired t-test: each subject contributes one stim and one sham value.
    test_result = ttest_rel(
        changes_df["delta_stimulation"],
        changes_df["delta_sham"],
        nan_policy="raise",
    )

    raw_df.to_csv(
        output_dir / "fake_tse_values.csv",
        index=False,
    )

    included_df.to_csv(
        output_dir / "included_complete_ses03_ses06.csv",
        index=False,
    )

    excluded_df.to_csv(
        output_dir / "excluded_subjects.csv",
        index=False,
    )

    long_df.to_csv(
        output_dir / "condition_long.csv",
        index=False,
    )

    changes_df.to_csv(
        output_dir / "change_scores.csv",
        index=False,
    )

    plot_condition_violin(
        long_df,
        "adaptation",
        output_dir / "adaptation_violin.png",
    )

    plot_condition_violin(
        long_df,
        "stimulation",
        output_dir / "stimulation_violin.png",
    )

    plot_condition_violin(
        long_df,
        "sham",
        output_dir / "sham_violin.png",
    )

    plot_change_distribution(
        changes_df["delta_stimulation"],
        "Distribution of stimulation change",
        output_dir / "stimulation_change_distribution.png",
    )

    plot_change_distribution(
        changes_df["delta_sham"],
        "Distribution of sham change",
        output_dir / "sham_change_distribution.png",
    )

    plot_paired_change(
        changes_df,
        output_dir / "paired_change_plot.png",
    )

    report = f"""
FAKE DATA ANALYSIS

Subjects in assignment table:
    {len(SUBJECT_TABLE)}

Included with complete ses03-ses06:
    {len(included_df)}

Excluded:
    {len(excluded_df)}

Included subjects missing ses01 or ses02:
    {included_df[["ses01", "ses02"]].isna().any(axis=1).sum()}

Mean stimulation change:
    {changes_df["delta_stimulation"].mean():.4f}

SD stimulation change:
    {changes_df["delta_stimulation"].std(ddof=1):.4f}

Mean sham change:
    {changes_df["delta_sham"].mean():.4f}

SD sham change:
    {changes_df["delta_sham"].std(ddof=1):.4f}

Mean paired difference, stimulation minus sham:
    {changes_df["stim_minus_sham"].mean():.4f}

Paired-samples t-test:
    t({len(changes_df) - 1}) = {test_result.statistic:.4f}
    p = {test_result.pvalue:.8f}
""".strip()

    print(report)

    (output_dir / "statistics.txt").write_text(
        report,
        encoding="utf-8",
    )


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(
        description=__doc__,
    )

    parser.add_argument(
        "--output-dir",
        default="fake_tse_analysis_output",
    )

    parser.add_argument(
        "--seed",
        type=int,
        default=20260724,
    )

    return parser.parse_args()


if __name__ == "__main__":
    arguments = parse_args()

    run_analysis(
        output_dir=arguments.output_dir,
        seed=arguments.seed,
    )

usage: ipykernel_launcher.py [-h] [--output-dir OUTPUT_DIR] [--seed SEED]
ipykernel_launcher.py: error: unrecognized arguments: --f=/run/user/1000/jupyter/runtime/kernel-v32cf507952e8d3a25bb80a2af80f5172267a2dc86.json


SystemExit: 2

/home/maria/miniconda3/envs/mri/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3756: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [ ]:
# Both experimental evenings are kept close.
common_evening = subject_baseline + rng.normal(
    0.0,
    evening_to_evening_sd,
)

ses03 = common_evening + rng.normal(
    0.0,
    evening_to_evening_sd / 2,
)

ses05 = common_evening + rng.normal(
    0.0,
    evening_to_evening_sd / 2,
)~~~~~~~~~~~~~~~~~~~~~~~

# Shared overnight/circadian effect.
circadian_change = rng.normal(
    circadian_mean,
    circadian_between_subject_sd,
)

# Additional stimulation effect.
stimulation_effect = rng.normal(
    stimulation_effect_mean,
    stimulation_effect_sd,
)

In [ ]:
ses04 = (
    ses03
    + circadian_change
    + stimulation_effect
    + overnight_noise
)

ses06 = (
    ses05
    + circadian_change
    + overnight_noise
)